In [1]:
print('hello world')

hello world


In [1]:
import h5py

input_file = 'https://s3.amazonaws.com/gtl-public-data/miten/rna004/bulk/PCA100286_20240626_2224_PAQ44197_1C_sequencing_run_06_26_24_RNA004_H9_tRNA_rsr_RT_9f10836d_4f621505.fast5'
output_file = 'test.fast5'
channel = '50'
with h5py.File(input_file, 'r', driver='ros3') as bulk_file:
    # Create a new HDF5 file
    with h5py.File(output_file, 'w') as subset_file:
        # Copy attributes from the original file to the new file
        for attr_name, attr_value in bulk_file.attrs.items():
            subset_file.attrs[attr_name] = attr_value

        # Copy required groups from the original file to the new file
        groups_to_copy = [
            # 'Device', 'Meta', 
                          'UniqueGlobalKey']
        for group_name in groups_to_copy:
            bulk_file.copy(group_name, subset_file)

        # Copy the 'StateData' group and its specified channel
        state_data_group = 'StateData'
        subset_file.create_group(state_data_group)
        bulk_file.copy(f'/{state_data_group}/Channel_{channel}', subset_file[state_data_group])
        
        # Extract the required slice of data from the 'Signal' dataset
        signal_data = bulk_file[f"/Raw/Channel_{channel}/Signal"][2000:4000]
        
        # Create the required groups in the subset file
        raw_group = subset_file.create_group('Raw')
        channel_50_group = raw_group.create_group(f'Channel_{channel}')
        
        # Create the dataset with the subset of data
        channel_50_group.create_dataset('Signal', data=signal_data)
        
        print(signal_data)


[265 267 273 ... 263 268 263]


In [1]:
from siren.utils.squiggletools import *

file_fn = 'test.fast5'
bulkfile = BulkFile(file_fn)

squiggle = bulkfile.fetch_squiggle('Channel_50:1-500')
annotation = bulkfile.fetch_annotation('Channel_50:1-500')
context = bulkfile.fetch_context()
tracking = bulkfile.fetch_tracking()
channels = bulkfile.list_channels()
channels

['Channel_50']

In [2]:
import pandas as pd
import h5py

# pd.DataFrame(annotation)
data_dtypes = h5py.check_dtype(enum=annotation.dtype['summary_state'])
data_dtypes = {v: k for k, v in data_dtypes.items()}
df_annotation = pd.DataFrame(annotation)
df_annotation['summary_state'] = df_annotation['summary_state'].map(data_dtypes)
# list(data_dtypes.values())
df_annotation

,acquisition_raw_index,analysis_raw_index,trigger_time,summary_state
0,7,3,0,unclassified
1,26,23,10,zero
2,26,26,23,unclassified
3,26,26,20,pending_mux_change
4,38,34,23,unclassified
5,57,57,43,unknown_positive
6,65,65,62,unclassified
7,65,65,53,pending_mux_change
8,72,72,62,unknown_positive
9,103,103,100,unclassified
